# Find Homography So Dope

Estimating the 3x3 homography that maps image 1 onto image 2 for each pair in a set of
6-image scenes. Scoring is the mean reprojection error of five reference points, the
four corners and the centre, mapped through the predicted and the true homography and
compared in normalised coordinates. The leaderboard displays
`100 * max(0, 1 - error / 0.2)`, so higher is better.

**Public score: 99.09285.**

This notebook contains the pipeline behind that submission (`24BCS10273_submission.csv`)
and reproduces it exactly. A second entry using photometric refinement was also
selected; it scored a little lower on the leaderboard despite scoring considerably
higher on the training set, and is described at the end.

Vimal Kumar Yadav — 24BCS10273

## Structure of the problem

Two properties of the data shape the whole solution.

**Half the scenes are illumination-only.** Their true homography is exactly the
identity: the camera never moved, only the lighting changed. Deciding this per scene
rather than per pair matters, because a scene's one badly matched pair then gets
overridden by its four good ones. Taking the median distance-from-identity across a
scene's five estimates separated all 28 training scenes cleanly — illumination scenes
topped out at 0.0048 and viewpoint scenes began at 0.196, so the threshold sits in a
gap roughly forty times wider than the quantity it is cutting.

**Transform magnitude grows with image index.** Image 6 is furthest from image 1, so a
1 -> 6 pair can be too wide to match directly even when 1 -> 5 and 5 -> 6 are both
easy. Those pairs are recovered by composing shorter hops through intermediate images,
with inlier count deciding whether the direct edge can be trusted as it stands.

In [1]:
import itertools
import os

import cv2
import numpy as np

NFEATURES = 8000
RATIO = 0.85
RANSAC_THRESH = 3.0
MAX_ITERS = 20000
CONFIDENCE = 0.9999
MIN_MATCHES = 4

MIN_DIRECT_INLIERS = 50

IDENTITY_SNAP = 0.01

GUIDED_RADII = (24.0, 10.0, 4.0)

CLAHE_CLIP = 3.0
CLAHE_GRID = (8, 8)

CORNERS = np.array([[0.0, 0.0], [1.0, 0.0], [1.0, 1.0], [0.0, 1.0], [0.5, 0.5]])
IDENTITY = np.eye(3, dtype=np.float64)

_scene_cache = {}

## Geometry helpers

`_rootsift` applies the Hellinger trick: L1-normalise each descriptor, then take the
square root, so an ordinary L2 matcher ends up computing Hellinger distance. It costs
nothing and matches noticeably better than raw SIFT.

`_degenerate` is deliberately permissive. A genuine homography in this data can be
non-convex with corners on either side of the horizon, so convexity and w-sign
consistency are treated as quality hints in `_clean` rather than as grounds for
discarding an estimate. Only visibly collapsed matrices, caught by the area ratio, are
rejected outright — a degenerate RANSAC consensus set folds the image rectangle into a
sliver, which the ratio detects.

In [2]:
def _rootsift(des):
    """L1-normalise then square-root: Hellinger distance under an L2 matcher."""
    if des is None:
        return None
    des = des / (des.sum(axis=1, keepdims=True) + 1e-7)
    return np.sqrt(des)


def _sanitise(H):
    if H is None:
        return None
    H = np.asarray(H, dtype=np.float64)
    if H.shape != (3, 3) or not np.all(np.isfinite(H)) or abs(H[2, 2]) < 1e-12:
        return None
    H = H / H[2, 2]
    if not np.all(np.isfinite(H)) or abs(float(np.linalg.det(H))) < 1e-12:
        return None
    return H


AREA_RATIO_MIN = 1e-3
AREA_RATIO_MAX = 1e3


def _quad(H, wh):
    """Image rectangle mapped through H: (corners, area ratio) or None."""
    if H is None:
        return None
    w, h = wh
    if min(w, h) <= 0:
        return None
    c = np.array([[0.0, 0.0, 1.0], [w, 0.0, 1.0], [w, h, 1.0], [0.0, h, 1.0]])
    p = c @ H.T
    if not np.all(np.isfinite(p)) or np.any(np.abs(p[:, 2]) < 1e-9):
        return None
    q = p[:, :2] / p[:, 2:3]
    if not np.all(np.isfinite(q)):
        return None
    x, y = q[:, 0], q[:, 1]
    area = 0.5 * abs(float(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))))
    return q, area / (w * h), np.sign(p[:, 2])


def _degenerate(H, wh):
    """Hard reject: only matrices that have visibly collapsed.

    Deliberately permissive about *extreme* transforms. A genuine ground-truth
    homography in this dataset can be non-convex with corners either side of the
    horizon (scene_010_1_5 is), so convexity and w-sign consistency are treated
    as quality hints in _clean(), never as grounds for throwing an estimate away.
    Degenerate RANSAC consensus sets collapse the rectangle to a sliver and are
    caught by the area ratio.
    """
    got = _quad(H, wh)
    if got is None:
        return True
    _, ratio, _ = got
    return not (AREA_RATIO_MIN < ratio < AREA_RATIO_MAX)


def _clean(H, wh):
    """Soft signal: the transform is convex and entirely in front of the camera."""
    got = _quad(H, wh)
    if got is None:
        return False
    q, ratio, signs = got
    if not np.all(signs == signs[0]):
        return False
    cross = []
    for t in range(4):
        a, b, c2 = q[t], q[(t + 1) % 4], q[(t + 2) % 4]
        u, v = b - a, c2 - b
        cross.append(u[0] * v[1] - u[1] * v[0])
    first = np.sign(cross[0])
    return bool(first != 0 and all(np.sign(v) == first for v in cross))

## The scene solver

Everything for one scene is computed once and cached, so the first pair of a scene is
expensive and the remaining four are effectively free.

The homography is fitted with MAGSAC rather than plain RANSAC, which is measurably more
accurate at the same threshold. On top of that sits **guided re-matching**: image 1's
keypoints are projected through the current estimate and re-matched within shrinking
radii of 24, 10 and 4 pixels, refitting at each step. This recovers correspondences that
Lowe's ratio test threw away as ambiguous — in repetitive scenes that is a large
fraction of them, and they are exactly the ones that pin down the wide-baseline pairs.

In [3]:
class _Scene:
    """Features, pairwise edges and per-target answers for one scene folder."""

    def __init__(self, folder):
        self.folder = folder
        self.sift = cv2.SIFT_create(nfeatures=NFEATURES)
        self.clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_GRID)
        self.matcher = cv2.BFMatcher(cv2.NORM_L2)
        self.idx = sorted(
            int(os.path.splitext(f)[0])
            for f in os.listdir(folder)
            if f.lower().endswith(".png") and os.path.splitext(f)[0].isdigit()
        )
        self._feat = {}
        self.edges = {}
        self._tried = set()
        self.answers = {}
        self.unresolved = set()
        self._solve()


    def _features(self, i):
        if i not in self._feat:
            img = cv2.imread(os.path.join(self.folder, f"{i}.png"), cv2.IMREAD_GRAYSCALE)
            if img is None:
                self._feat[i] = ((), None, (0, 0))
            else:
                kp, des = self.sift.detectAndCompute(self.clahe.apply(img), None)
                h, w = img.shape[:2]
                self._feat[i] = (kp, _rootsift(des), (w, h))
        return self._feat[i]

    def size(self, i):
        return self._features(i)[2]

    def _ratio_matches(self, des1, des2):
        if des1 is None or des2 is None or len(des1) < 2 or len(des2) < 2:
            return []
        out = {}
        for pair in self.matcher.knnMatch(des1, des2, k=2):
            if len(pair) < 2:
                continue
            m, n = pair
            if m.distance < RATIO * n.distance:
                out[m.queryIdx] = m.trainIdx
        return list(out.items())

    def _fit(self, kp1, kp2, matches):
        if len(matches) < MIN_MATCHES:
            return None, 0
        src = np.float32([kp1[q].pt for q, _ in matches]).reshape(-1, 1, 2)
        dst = np.float32([kp2[t].pt for _, t in matches]).reshape(-1, 1, 2)
        H, mask = cv2.findHomography(
            src, dst, cv2.USAC_MAGSAC, ransacReprojThreshold=RANSAC_THRESH,
            maxIters=MAX_ITERS, confidence=CONFIDENCE,
        )
        H = _sanitise(H)
        if H is None:
            return None, 0
        return H, (0 if mask is None else int(mask.sum()))

    def _edge(self, i, j):
        """Match i -> j once, memoised, storing the inverse direction too."""
        if (i, j) in self.edges or (i, j) in self._tried:
            return self.edges.get((i, j))
        self._tried.add((i, j))
        self._tried.add((j, i))
        kp1, des1, _ = self._features(i)
        kp2, des2, _ = self._features(j)
        H, n = self._fit(kp1, kp2, self._ratio_matches(des1, des2))
        if H is not None and _degenerate(H, self.size(i)):
            H = None
        if H is not None:
            self.edges[(i, j)] = (H, n)
            inv = _sanitise(np.linalg.inv(H))
            if inv is not None:
                self.edges[(j, i)] = (inv, n)
        return self.edges.get((i, j))


    def _corner_shift(self, H, i, j):
        """Distance the five reference points move, in the metric's own units."""
        w1, h1 = self.size(i)
        w2, h2 = self.size(j)
        if min(w1, h1, w2, h2) <= 0:
            return np.inf
        pts = np.column_stack([CORNERS[:, 0] * w1, CORNERS[:, 1] * h1, np.ones(len(CORNERS))])
        a = pts @ H.T
        b = pts @ IDENTITY.T
        if np.any(np.abs(a[:, 2]) < 1e-12):
            return np.inf
        an = np.column_stack([a[:, 0] / a[:, 2] / w2, a[:, 1] / a[:, 2] / h2])
        bn = np.column_stack([b[:, 0] / b[:, 2] / w2, b[:, 1] / b[:, 2] / h2])
        return float(np.mean(np.linalg.norm(an - bn, axis=1)))

    def _guided_refine(self, i, j, H):
        """Re-match under the current H within shrinking radii, refitting each time."""
        kp1, des1, _ = self._features(i)
        kp2, des2, _ = self._features(j)
        if H is None or des1 is None or des2 is None or not len(kp1) or not len(kp2):
            return H
        p1 = np.array([k.pt for k in kp1], dtype=np.float64)
        p2 = np.array([k.pt for k in kp2], dtype=np.float64)
        best = H
        for radius in GUIDED_RADII:
            with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
                proj = np.column_stack([p1, np.ones(len(p1))]) @ best.T
            if not np.all(np.isfinite(proj)):
                break
            w = proj[:, 2]
            ok = np.abs(w) > 1e-12
            if ok.sum() < MIN_MATCHES:
                break
            proj = proj[ok, :2] / w[ok, None]
            src_idx = np.flatnonzero(ok)

            d = np.linalg.norm(proj[:, None, :] - p2[None, :, :], axis=2) \
                if len(p2) * len(proj) <= 4_000_000 else None
            if d is None:
                break
            nearest = np.argmin(d, axis=1)
            keep = d[np.arange(len(proj)), nearest] < radius
            if keep.sum() < MIN_MATCHES:
                break
            matches = list(zip(src_idx[keep], nearest[keep]))
            cand, n = self._fit(kp1, kp2, matches)
            if cand is None or n < MIN_MATCHES:
                break
            best = cand
        return best


    def _candidates(self, target):
        """Every route 1 -> target, as (H, bottleneck_inliers, path)."""
        out = []
        direct = self.edges.get((1, target))
        if direct is not None:
            out.append((direct[0], direct[1], (1, target)))
        others = [n for n in self.idx if n not in (1, target)]
        for r in range(1, min(3, len(others)) + 1):
            for mid in itertools.permutations(others, r):
                path = (1,) + mid + (target,)
                edges = [(path[t], path[t + 1]) for t in range(len(path) - 1)]
                for e in edges:
                    self._edge(*e)
                if any(e not in self.edges for e in edges):
                    continue
                H = IDENTITY.copy()
                for e in edges:
                    H = self.edges[e][0] @ H
                H = _sanitise(H)
                if H is None or _degenerate(H, self.size(1)):
                    continue
                out.append((H, min(self.edges[e][1] for e in edges), path))
        return out

    def _solve(self):
        targets = [k for k in self.idx if k != 1]

        direct = {k: self._edge(1, k) for k in targets}

        shifts = [self._corner_shift(e[0], 1, k)
                  for k, e in direct.items() if e is not None]
        if not shifts or float(np.median(shifts)) < IDENTITY_SNAP:
            for k in targets:
                self.answers[k] = IDENTITY.copy()
            return

        for k in targets:
            e = direct.get(k)
            if e is not None and e[1] >= MIN_DIRECT_INLIERS and not _degenerate(e[0], self.size(1)):
                best = e[0]
            else:
                cands = self._candidates(k)
                if cands:
                    best = max(cands, key=lambda c: (_clean(c[0], self.size(1)),
                                                     c[1], -len(c[2])))[0]
                else:
                    best = e[0] if e is not None else None

            if best is None:
                self.unresolved.add(k)
                self.answers[k] = IDENTITY.copy()
                continue

            refined = self._guided_refine(1, k, best)
            if refined is not None and not _degenerate(refined, self.size(1)):
                best = refined
            self.answers[k] = best

## The prediction entry point

The benchmark harness imports this function directly, so the signature is fixed. Scene
state is cached by folder, which is why the per-pair runtime is far below the per-scene
cost.

In [4]:
def predict(image_1_path: str, image_2_path: str) -> np.ndarray:
    try:
        folder = os.path.dirname(image_1_path)
        if folder not in _scene_cache:
            _scene_cache[folder] = _Scene(folder)
        scene = _scene_cache[folder]
        target = int(os.path.splitext(os.path.basename(image_2_path))[0])
        H = scene.answers.get(target)
        if H is not None:
            return np.asarray(H, dtype=np.float64)
    except Exception:
        pass
    return IDENTITY.copy()

## Generating the submission

In [5]:
import csv, os, time
import numpy as np

COLS = ["pair_id", "h11", "h12", "h13", "h21", "h22", "h23", "h31", "h32"]

def build(pairs_csv, data_dir, out_path):
    with open(pairs_csv, newline="") as f:
        pairs = list(csv.DictReader(f))
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    t0 = time.perf_counter()
    identity_count = 0
    with open(out_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(COLS)
        for row in pairs:
            H = np.asarray(predict(os.path.join(data_dir, row["image_1"]),
                                   os.path.join(data_dir, row["image_2"])), dtype=np.float64)
            if np.allclose(H, np.eye(3)):
                identity_count += 1
            H = H / H[2, 2]
            w.writerow([row["pair_id"]] + [f"{v:.12g}" for v in
                       (H[0,0], H[0,1], H[0,2], H[1,0], H[1,1], H[1,2], H[2,0], H[2,1])])
    print(f"wrote {out_path}  ({len(pairs)} rows, {time.perf_counter()-t0:.1f}s)")
    print(f"identity predictions: {identity_count}/{len(pairs)}")

build("test.csv", "data/test", "out/notebook_submission.csv")

wrote out/notebook_submission.csv  (60 rows, 4.5s)
identity predictions: 30/60


Exactly half the test predictions come back as the identity. That is the
illumination split showing through: 6 of the 12 test scenes are illumination-only, and
the scene-level classifier caught all of them.

In [6]:
import filecmp
print("matches the submitted file:",
      filecmp.cmp("out/notebook_submission.csv", "out/24BCS10273_submission.csv", shallow=False))

matches the submitted file: True


## Local validation against the training ground truth

`train.csv` ships the true homographies, so the competition metric can be computed
locally before trusting anything.

In [7]:
def leaderboard_score(pred_csv, truth_csv, data_dir):
    truth = {r["pair_id"]: r for r in csv.DictReader(open(truth_csv))}
    def H_of(r):
        v = [float(r[k]) for k in ("h11","h12","h13","h21","h22","h23","h31","h32")]
        return np.array([[v[0],v[1],v[2]],[v[3],v[4],v[5]],[v[6],v[7],1.0]])
    dims = {}
    def size(scene, i):
        key = (scene, i)
        if key not in dims:
            img = cv2.imread(f"{data_dir}/{scene}/{i}.png")
            h, w = img.shape[:2]
            dims[key] = (w, h)
        return dims[key]
    errs, ids = [], []
    for r in csv.DictReader(open(pred_csv)):
        pid = r["pair_id"]
        scene = pid.rsplit("_1_", 1)[0]
        target = int(pid.rsplit("_", 1)[1])
        w1, h1 = size(scene, 1)
        w2, h2 = size(scene, target)
        pts = np.column_stack([CORNERS[:,0]*w1, CORNERS[:,1]*h1, np.ones(len(CORNERS))])
        got = []
        for H in (H_of(truth[pid]), H_of(r)):
            p = pts @ H.T
            got.append(np.column_stack([p[:,0]/p[:,2]/w2, p[:,1]/p[:,2]/h2]))
        errs.append(float(np.mean(np.linalg.norm(got[0]-got[1], axis=1))))
        ids.append(pid)
    err = float(np.mean(errs))
    return err, 100 * max(0.0, 1 - err / 0.2), errs, ids

build("train.csv", "data/train", "out/notebook_train.csv")
err, score, per_pair, ids = leaderboard_score("out/notebook_train.csv", "train.csv", "data/train")
print(f"\nreprojection error  {err:.6f}   (lower is better)")
print(f"leaderboard score   {score:.5f}   (higher is better)")

wrote out/notebook_train.csv  (140 rows, 13.7s)
identity predictions: 70/140



reprojection error  0.003034   (lower is better)
leaderboard score   98.48291   (higher is better)


### Where the remaining error sits

The error is nowhere near evenly spread, and seeing how concentrated it is explains the
shape of the whole scoreboard.

In [8]:
order = sorted(range(len(per_pair)), key=lambda i: -per_pair[i])
total = sum(per_pair)
run = 0.0
print("worst pairs:")
for i in order[:6]:
    run += per_pair[i]
    print(f"  {ids[i]:18s} {per_pair[i]:.4f}   {100*per_pair[i]/total:5.1f}% of total   "
          f"cumulative {100*run/total:5.1f}%")

worst pairs:
  scene_010_1_5      0.1791    42.2% of total   cumulative  42.2%
  scene_006_1_6      0.0548    12.9% of total   cumulative  55.1%
  scene_038_1_6      0.0237     5.6% of total   cumulative  60.6%
  scene_033_1_6      0.0194     4.6% of total   cumulative  65.2%
  scene_033_1_5      0.0188     4.4% of total   cumulative  69.6%
  scene_038_1_5      0.0186     4.4% of total   cumulative  74.0%


## Why one pair dominates

`scene_010_1_5` is worth looking at directly.

Its homography maps a corner of image 1 to a point whose `w` is about `-0.1` — past the
horizon, behind the camera — and the metric divides by that `w`. So an estimate that
matches the ground truth to four decimal places still scores badly here: four of its
five reference points land almost exactly right, and the fifth is amplified roughly a
hundredfold.

This is the ceiling of the whole task. The pair is not mismatched and no amount of
better matching fixes it; the metric is simply hypersensitive at that geometry.

In [9]:
truth = {r["pair_id"]: r for r in csv.DictReader(open("train.csv"))}
def H_of(r):
    v = [float(r[k]) for k in ("h11","h12","h13","h21","h22","h23","h31","h32")]
    return np.array([[v[0],v[1],v[2]],[v[3],v[4],v[5]],[v[6],v[7],1.0]])
pred = {r["pair_id"]: H_of(r) for r in csv.DictReader(open("out/notebook_train.csv"))}

pid = "scene_010_1_5"
a = cv2.imread("data/train/scene_010/1.png"); h1, w1 = a.shape[:2]
b = cv2.imread("data/train/scene_010/5.png"); h2, w2 = b.shape[:2]
pts = np.column_stack([CORNERS[:,0]*w1, CORNERS[:,1]*h1, np.ones(5)])
for name, H in (("ground truth", H_of(truth[pid])), ("predicted", pred[pid])):
    p = pts @ H.T
    print(f"{name:13s} w at the five points: {np.array2string(p[:,2], precision=4)}")
pa = pts @ H_of(truth[pid]).T; pb = pts @ pred[pid].T
na = np.column_stack([pa[:,0]/pa[:,2]/w2, pa[:,1]/pa[:,2]/h2])
nb = np.column_stack([pb[:,0]/pb[:,2]/w2, pb[:,1]/pb[:,2]/h2])
d = np.linalg.norm(na-nb, axis=1)
print(f"\nper-point error: {np.array2string(d, precision=4)}")
print(f"one point contributes {100*d.max()/d.sum():.0f}% of this pair's error")

ground truth  w at the five points: [ 1.     -0.1689 -0.1085  1.0604  0.4457]
predicted     w at the five points: [ 1.     -0.1633 -0.0973  1.066   0.4513]

per-point error: [0.0058 0.0953 0.7892 0.0039 0.0012]
one point contributes 88% of this pair's error


## Runtime

Scene state is cached, so only the first pair of a scene does real work and the median
per-pair time is effectively zero. The cache is cleared first so this measures a cold
run rather than the state left behind by the cells above. Two warm-up pairs are excluded
and timing uses `time.perf_counter()`, matching the official harness.

In [10]:
_scene_cache.clear()

pairs = list(csv.DictReader(open("test.csv")))
paths = [(os.path.join("data/test", r["image_1"]), os.path.join("data/test", r["image_2"]))
         for r in pairs]
for p1, p2 in paths[:2]:
    predict(p1, p2)
per = []
for p1, p2 in paths:
    t = time.perf_counter(); predict(p1, p2); per.append(time.perf_counter() - t)
per = np.array(per)
print(f"pairs          {len(per)}")
print(f"total          {per.sum():.2f}s")
print(f"mean per pair  {per.mean():.3f}s")
print(f"median         {np.median(per):.3f}s")
print(f"slowest pair   {per.max():.3f}s")

pairs          60
total          4.62s
mean per pair  0.077s
median         0.000s
slowest pair   1.097s


## What else was tried

Recording the approaches that did not make it in, since the negative results are what
pointed at what mattered.

| approach | train | leaderboard |
|---|---|---|
| this notebook | 98.48 | **99.09285** |
| route consensus across composed paths | 98.29 | not submitted |
| consensus used only as a veto | 98.44 | not submitted |
| joint bundle refinement over the scene graph | 98.73 | 98.81967 |
| bundle gated on evidence strength | 98.52 | not submitted |
| photometric (ECC) refinement, accepted on correlation | 99.38 | 98.23892 |
| photometric refinement, accepted on inlier agreement | 99.38 | 99.00346 |
| the above with a tighter residual tolerance | 99.28 | 99.05830 |
| 16k / 32k SIFT features | 98.70 | not submitted |

Two things came out of this.

**Everything that only rearranges sparse estimates lands in the same place.** Route
selection, consensus and bundle adjustment all inherit the same half-pixel keypoint
localisation error, which is why they finished within noise of one another. Photometric
refinement was the only change that moved that floor, lifting the training score by
nearly a full point.

**A threshold tuned against the training failures does not transfer.** The bundle gates
and the first photometric gate were both chosen by looking at which training scenes
diverged, and both lost on the leaderboard. The training set showed no regression worse
than 0.005 for the photometric gate, yet it moved one test pair by 0.107 — that failure
mode simply does not occur in any of the fourteen training viewpoint scenes.

That is also why the plain pipeline in this notebook is the one that scored highest.
Photometric refinement is clearly better on training data, by nearly a point, but its
gain there comes overwhelmingly from rescuing near-horizon pairs like `scene_010_1_5`,
and the test set contains only one pair with that geometry. With little to rescue, what
remained on the leaderboard was its downside. Both were selected as final submissions
for that reason.